# GR5069 Take Home Exercise 2
### *Xuejing Yan*

1. What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

- Logic: Since the question asked for pit stop for each race, so I will need to load the pitstops dataset using spark. Then I will need to group raceId and driverId to get the average time for each driver in each race. In order to get the fastest and slowest pit stop time each race, I will group pit stop data by raceId only and get the min/max value. 


In [0]:
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)


In [0]:
display(df_pitstops)

In [0]:

display(df_pitstops.filter(col("milliseconds").isNull()))

In [0]:
from pyspark.sql.functions import avg, round

driver_race_avg = (
    df_pitstops
    .groupBy("raceId", "driverId")
    .agg(round(avg("milliseconds"), 2).alias("avg_ps"))
)

In [0]:
display(driver_race_avg)

In [0]:
from pyspark.sql.functions import min, max
race_pit_summary = (df_pitstops.groupby("raceId")
                    .agg(min("milliseconds").alias("min_ps"),
                         max("milliseconds").alias("max_ps"))
                    )
display(race_pit_summary)

- I used `groupBy()` function to group raceId and driverId together, then use the function `avg()` to computes the average of all their pit stops in that race. To find the fastest and slowest pit stop in each race overall, I grouped only by `raceId` and used `min("milliseconds")` and `max("milliseconds")`. These return a dataframe with the smallest and largest pit stop times recorded in that race. 

2. Rank order by finishing position the average time spent at the pit stop in each race.

- Logic: In order to get finishing position, i will need to load the `results.csv` and combine with `driver_race_avg` I created from last question. I will join this result with the `results` dataset using `raceId` and `driverId` so that I could associate each driver’s average pit stop time with their finishing position in that race. After that, I rank the drivers within each race based on their finishing position.

In [0]:
results = (spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True))

display(results)

In [0]:
from pyspark.sql.functions import col
results_clean = (
    results
    .filter(col("position").isNotNull())
    .filter(col("position") != "\\N")
    .withColumn("position", col("position").cast("int"))
)

display(results_clean)

In [0]:

pitstops_position = (
    driver_race_avg
    .join(results_clean, on=["raceId", "driverId"], how="inner")
    .select("raceId", "driverId", "position", "avg_ps")
    .orderBy(col("raceId").asc(), col("position").asc())
)

display(pitstops_position)

- I first loaded the `results.csv` and cleaned it by filtering out rows where `position` was null or equal to `\N`. Because I saw the dataframe have \N value. I then converted `position` to an integer using `.cast("int")` so it could be sorted numerically.

- Next, I joined the average pit stop I computed before table with the cleaned finishing position table using `.join(..., on=["raceId", "driverId"], how="inner")`. I only selected relevant columns, and sorted the output by `raceId` and `position` so that each race is displayed in finishing order.